# Welcome to Colab!

<div class="markdown-google-sans">
  <h2>What is Colab?</h2>
</div>

Colab, or "Colaboratory", allows you to write and execute Python in your browser, with
- Zero configuration required
- Access to GPUs free of charge
- Easy sharing

Whether you're a **student**, a **data scientist** or an **AI researcher**, Colab can make your work easier. Watch [Introduction to Colab](https://www.youtube.com/watch?v=inN8seMm7UI) or [Colab Features You May Have Missed](https://www.youtube.com/watch?v=rNgswRZ2C1Y) to learn more, or just get started below!

In [2]:
import sqlite3
import random
from datetime import datetime, timedelta

random.seed(42)

conn = sqlite3.connect('clinic.db')
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON;")
cursor.execute("DROP TABLE IF EXISTS appointments;")
cursor.execute("DROP TABLE IF EXISTS patients;")
cursor.execute("DROP TABLE IF EXISTS doctors;")

cursor.execute("""
CREATE TABLE doctors (
    doctor_id INTEGER PRIMARY KEY AUTOINCREMENT,
    doctor_name TEXT NOT NULL,
    department TEXT NOT NULL  -- Nominal Data Category
);
""")

cursor.execute("""
CREATE TABLE patients (
    patient_id INTEGER PRIMARY KEY AUTOINCREMENT,
    full_name TEXT NOT NULL,
    contact_email TEXT UNIQUE,  -- Kept nullable for missing data realism
    birth_year INTEGER NOT NULL CHECK(birth_year >= 1940 AND birth_year <= 2024), -- Interval Data
    insurance_status TEXT NOT NULL CHECK(insurance_status IN ('Insured', 'Uninsured', 'State-Aided')) -- Nominal Data
);
""")

cursor.execute("""
CREATE TABLE appointments (
    appointment_id INTEGER NOT NULL,
    patient_id INTEGER NOT NULL,
    doctor_id INTEGER NOT NULL,
    triage_priority TEXT NOT NULL CHECK(triage_priority IN ('Low', 'Medium', 'High', 'Critical')), -- Ordinal Data
    consultation_fee REAL NOT NULL CHECK(consultation_fee >= 0), -- Ratio Data
    waiting_time_minutes INTEGER NOT NULL CHECK(waiting_time_minutes >= 0), -- Ratio Data
    appointment_date TEXT NOT NULL,
    PRIMARY KEY (appointment_id, patient_id, doctor_id), -- Composite Primary Key
    FOREIGN KEY (patient_id) REFERENCES patients(patient_id) ON DELETE CASCADE,
    FOREIGN KEY (doctor_id) REFERENCES doctors(doctor_id) ON DELETE CASCADE
);
""")

conn.commit()
print("Database schema created successfully with strict NOT NULL constraints.")

departments = ["Cardiology", "Dermatology", "Pediatrics", "Orthopedics", "General Medicine"]
doctors_data = []
for i in range(1, 16):
    name = f"Dr. Specialist {i}"
    dept = random.choice(departments) # Nominal Data
    doctors_data.append((i, name, dept))

cursor.executemany("INSERT INTO doctors VALUES (?, ?, ?)", doctors_data)

first_names = ["Aarav", "Priya", "Rohan", "Ananya", "Vikram", "Neha", "John", "Emma", "Liam", "Olivia"]
last_names = ["Sharma", "Verma", "Patel", "Reddy", "Gupta", "Smith", "Johnson", "Brown"]
statuses = ["Insured", "Uninsured", "State-Aided"]

patients_data = []
for p_id in range(1, 101):
    fname = random.choice(first_names)
    lname = random.choice(last_names)
    name = f"{fname} {lname}"

    email = f"{fname.lower()}.{lname.lower()}{p_id}@example.com" if p_id % 12 != 0 else None

    birth_yr = random.randint(1950, 2015) # Interval Data
    status = random.choice(statuses)
    patients_data.append((p_id, name, email, birth_yr, status))

cursor.executemany("INSERT INTO patients VALUES (?, ?, ?, ?, ?)", patients_data)
conn.commit()

appointments_data = []
start_date = datetime(2024, 1, 1)
priorities = ["Low", "Medium", "High", "Critical"] # Ordinal Data

for app_id in range(1, 1205):
    pat_id = random.randint(1, 100)
    doc_id = random.randint(1, 15)
    priority = random.choice(priorities)

    fee = round(random.uniform(30.0, 350.0), 2) # Ratio Data
    wait_time = random.randint(0, 90) # Ratio Data

    rand_days = random.randint(0, 700)
    app_date = (start_date + timedelta(days=rand_days)).strftime('%Y-%m-%d')

    appointments_data.append((app_id, pat_id, doc_id, priority, fee, wait_time, app_date))

for i in range(30):
    original_row = appointments_data[i]
    new_app_id = 1205 + i
    duplicate_row = (
        new_app_id,
        original_row[1], # patient_id
        original_row[2], # doctor_id
        original_row[3], # priority
        original_row[4], # fee
        original_row[5], # wait_time
        original_row[6]  # date
    )
    appointments_data.append(duplicate_row)

cursor.executemany("INSERT INTO appointments VALUES (?, ?, ?, ?, ?, ?, ?)", appointments_data)
conn.commit()

cursor.execute("SELECT COUNT(*) FROM appointments;")
total_rows = cursor.fetchone()[0]
print(f"Data generation complete! Total rows successfully inserted into appointments table: {total_rows}")

conn.close()

try:
    from google.colab import files
    files.download('clinic.db')
    print("Database download triggered via Google Colab.")
except ImportError:
    print("Running locally: 'clinic.db' has been saved successfully to the project folder.")

Database schema created successfully with strict NOT NULL constraints.
Data generation complete! Total rows successfully inserted into appointments table: 1234


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Database download triggered via Google Colab.
